In [1]:
from google.cloud import bigquery
import pandas as pd

# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 1: publications — one row per paper
# ═══════════════════════════════════════════════════════════════════════════════
classif = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["int_id", "micro", "meso", "macro"],
)
meta = pd.read_csv(
    "cwts_output/pub_metadata.txt",
    sep="\t",
    header=None,
    names=["int_id", "pub_id", "is_frontiers", "journal", "date", "title"],
    on_bad_lines="warn",
)
publications = classif.merge(meta, on="int_id")
publications = publications[
    [
        "pub_id",
        "int_id",
        "journal",
        "date",
        "title",
        "is_frontiers",
        "micro",
        "meso",
        "macro",
    ]
]
print(f"publications: {len(publications):,} rows")
# publications.to_csv("bq_publications.csv", index=False)
# or: publications.to_gbq("your_dataset.publications", project_id="...", if_exists="replace")
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 2: citation_links — network edges
# ═══════════════════════════════════════════════════════════════════════════════
cit_links = pd.read_csv(
    "cwts_output/cit_links.txt",
    sep="\t",
    header=None,
    names=["int_id1", "int_id2", "weight"],
)
# Optionally add pub_ids for easier joins
id_map = publications[["int_id", "pub_id"]].drop_duplicates()
citation_links = (
    cit_links.merge(
        id_map.rename(columns={"int_id": "int_id1", "pub_id": "pub_id1"}), on="int_id1"
    ).merge(
        id_map.rename(columns={"int_id": "int_id2", "pub_id": "pub_id2"}), on="int_id2"
    )
)[["pub_id1", "pub_id2", "weight"]]
print(f"citation_links: {len(citation_links):,} rows")
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 3: cluster_taxonomy — taxonomy mapping from AA-978
# ═══════════════════════════════════════════════════════════════════════════════
df_out = pd.read_csv(
    "cwts_output/community_cluster_assignments.csv"
)  # or whatever you named it

# Assuming df_out is already in memory from the AA-978 notebook
cluster_taxonomy = df_out[
    [
        "community_id",
        "community_name",
        "n_community_papers",
        "tier",
        "cluster_rank",
        "cluster_key",
        "cluster_name",
        "cluster_level",
        "n_l2_covered",
        "l2_coverage",
        "article_share",
        "match_mode",
        "llm_confidence",
        "llm_rationale",
        "llm_reasoning",
    ]
].rename(
    columns={"community_id": "macro_cluster"}
)  # rename to match publications table
print(f"cluster_taxonomy: {len(cluster_taxonomy):,} rows")
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 4: cluster_primary_labels — simplified one-row-per-cluster lookup
# ═══════════════════════════════════════════════════════════════════════════════
cluster_labels = df_out[(df_out["tier"] == "core") & (df_out["cluster_rank"] == 1)][
    ["community_id", "cluster_name", "cluster_level", "article_share", "match_mode"]
].rename(columns={"community_id": "macro_cluster", "cluster_name": "primary_taxonomy"})
print(f"cluster_labels: {len(cluster_labels):,} rows")
# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print("\n=== Tables ready for BigQuery ===")
print(f"  publications:     {len(publications):,} rows  (pub_id, journal, clusters)")
print(f"  citation_links:   {len(citation_links):,} rows  (pub_id1, pub_id2, weight)")
print(f"  cluster_taxonomy: {len(cluster_taxonomy):,} rows  (full taxonomy mapping)")
print(f"  cluster_labels:   {len(cluster_labels):,} rows  (primary label per cluster)")

/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 2624071: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 3795906: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 5919456: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 7692718: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 8114851: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 13243549: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 13707634: expected 6 fields, saw 7

  meta = pd.read_csv(
/var/tmp/ipykernel_47257/1860514591.py:13: ParserWarning: Skipping line 14147173: expected 6 fi

SystemError: <method 'rfind' of 'str' objects> returned a result with an exception set

In [ ]:
# # Run once in terminal if not already done:
# # gcloud auth application-default login


# project = "ocean-tech-adv-analytics-c-tfs"
# dataset = "scope_drift"

# client = bigquery.Client(project="ocean-tech-adv-analytics-c-tfs")

# # Create dataset if it doesn't exist
# dataset_id = "ocean-tech-adv-analytics-c-tfs.scope_drift"
# # client.create_dataset(dataset_id, exists_ok=True)
# # print("Dataset ready")

# publications.to_gbq(f"{dataset}.publications", project_id=project, if_exists="replace")
# citation_links.to_gbq(
#     f"{dataset}.citation_links", project_id=project, if_exists="replace"
# )
# cluster_taxonomy.to_gbq(
#     f"{dataset}.cluster_taxonomy", project_id=project, if_exists="replace"
# )
# cluster_labels.to_gbq(
#     f"{dataset}.cluster_labels", project_id=project, if_exists="replace"
# )

In [1]:
## This is to write the raw pub data from cell 1 
import pandas as pd
# Load raw files
pub_metadata = pd.read_csv(
    "cwts_output/pub_metadata.txt", 
    sep="\t", 
    header=None,
    names=["int_id", "pub_id", "is_frontiers", "journal", "date", "title"],
    on_bad_lines="warn"
)

pubs = pd.read_csv(
    "cwts_output/pubs.txt",
    sep="\t",
    header=None,
    names=["int_id", "core_pub"]
)

cits = pd.read_csv(
    "cwts_output/cit_links.txt",
    sep="\t",
    header=None,
    names=["int_id1", "int_id2", "weight"]
)
# Upload to BigQuery
project = "ocean-tech-adv-analytics-c-tfs"  # your project
dataset = "scope_drift_raw"  # change to your dataset name
pub_metadata.to_gbq(
    f"{dataset}.pub_metadata_raw",
    project_id=project,
    if_exists="replace"
)
print(f"Uploaded pub_metadata: {len(pub_metadata):,} rows")
pubs.to_gbq(
    f"{dataset}.pubs_raw",
    project_id=project,
    if_exists="replace"
)
print(f"Uploaded pubs: {len(pubs):,} rows")


cits.to_gbq(
    f"{dataset}.cit_links_raw",
    project_id=project,
    if_exists="replace"
)
print(f"Uploaded cit_links: {len(cits):,} rows")

/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 2624071: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 3795906: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 5919456: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 7692718: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 8114851: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 13243549: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: ParserWarning: Skipping line 13707634: expected 6 fields, saw 7

  pub_metadata = pd.read_csv(
/var/tmp/ipykernel_64694/3000255868.py:4: Pars

Uploaded pub_metadata: 30,830,359 rows


100%|██████████| 1/1 [00:00<00:00, 19418.07it/s]
/var/tmp/ipykernel_64694/3000255868.py:42: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  cits.to_gbq(


Uploaded pubs: 31,125,481 rows


100%|██████████| 1/1 [00:00<00:00, 24105.20it/s]

Uploaded cit_links: 588,708,600 rows


In [3]:
from google.cloud import bigquery
import pandas as pd
# Push classification links
classif = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["int_id", "micro", "meso", "macro"]
)
# Upload to BigQuery
project = "ocean-tech-adv-analytics-c-tfs"  # your project
dataset = "scope_drift_raw"  # change to your dataset name
classif.to_gbq(
    f"{dataset}.pub_classification_raw",
    project_id=project,
    if_exists="replace"
)

/var/tmp/ipykernel_3586/3447851645.py:13: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  classif.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 17119.61it/s]
